In [1]:
!pip install ucimlrepo -q

In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

dataset = fetch_ucirepo(id=697)
df = dataset.data.features.copy()
df["Target"] = dataset.data.targets["Target"]

print(df.shape)
df.head()

(4424, 37)


,Marital Status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


In [3]:
df["Dropout"] = (df["Target"] == "Dropout").astype(int)
print(df["Dropout"].value_counts())
print(df["Dropout"].value_counts(normalize=True).round(3))

Dropout
0    3003
1    1421
Name: count, dtype: int64
Dropout
0    0.679
1    0.321
Name: proportion, dtype: float64


In [4]:
print("Valeurs manquantes :", df.isna().sum().sum())
df.info()

Valeurs manquantes : 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 38 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital Status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance                      4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nacionality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-null   int64  
 9   Father's qualifica

In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Marital Status,4424.0,1.178571,0.605747,1.00,1.00,1.000000,1.000000,6.000000
Application mode,4424.0,18.669078,17.484682,1.00,1.00,17.000000,39.000000,57.000000
Application order,4424.0,1.727848,1.313793,0.00,1.00,1.000000,2.000000,9.000000
Course,4424.0,8856.642631,2063.566416,33.00,9085.00,9238.000000,9556.000000,9991.000000
Daytime/evening attendance,4424.0,0.890823,0.311897,0.00,1.00,1.000000,1.000000,1.000000
Previous qualification,4424.0,4.577758,10.216592,1.00,1.00,1.000000,1.000000,43.000000
Previous qualification (grade),4424.0,132.613314,13.188332,95.00,125.00,133.100000,140.000000,190.000000
Nacionality,4424.0,1.873192,6.914514,1.00,1.00,1.000000,1.000000,109.000000
Mother's qualification,4424.0,19.561935,15.603186,1.00,2.00,19.000000,37.000000,44.000000
Father's qualification,4424.0,22.275316,15.343108,1.00,3.00,19.000000,37.000000,44.000000


In [6]:
df.drop(columns=["Target", "Dropout"]).nunique().sort_values().head(25)

Daytime/evening attendance                         2
Displaced                                          2
Debtor                                             2
Educational special needs                          2
International                                      2
Scholarship holder                                 2
Gender                                             2
Tuition fees up to date                            2
Marital Status                                     6
Application order                                  8
Inflation rate                                     9
Curricular units 2nd sem (without evaluations)    10
GDP                                               10
Unemployment rate                                 10
Curricular units 1st sem (without evaluations)    11
Previous qualification                            17
Course                                            17
Application mode                                  18
Curricular units 2nd sem (credited)           

In [7]:
import plotly.express as px

col = "Tuition fees up to date"
taux = df.groupby(col)["Dropout"].mean().reset_index()
px.bar(taux, x=col, y="Dropout", title=f"Taux de décrochage selon : {col}").show()

In [8]:
corr = df.drop(columns="Target").corr()["Dropout"].drop("Dropout").sort_values()
print("Les plus négatives (protègent du décrochage) :")
print(corr.head(5))
print("\nLes plus positives (augmentent le décrochage) :")
print(corr.tail(5))

Les plus négatives (protègent du décrochage) :
Curricular units 2nd sem (grade)      -0.571792
Curricular units 2nd sem (approved)   -0.569500
Curricular units 1st sem (grade)      -0.480669
Curricular units 1st sem (approved)   -0.479112
Tuition fees up to date               -0.429149
Name: Dropout, dtype: float64

Les plus positives (augmentent le décrochage) :
Marital Status       0.093712
Application mode     0.198458
Gender               0.203983
Debtor               0.229407
Age at enrollment    0.254215
Name: Dropout, dtype: float64


In [9]:
# X = variables explicatives, y = cible binaire
X = df.drop(columns=["Target", "Dropout"])
y = df["Dropout"]

# Catégories codées par des entiers (le code n'a aucun ordre réel)
categorical_cols = [
    "Marital Status", "Application mode", "Course", "Previous qualification",
    "Nacionality", "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation",
]

# Variables binaires (0/1) : on les laisse telles quelles
binary_cols = [c for c in X.columns if X[c].nunique() == 2]

# Le reste est numérique : on le normalisera
numeric_cols = [c for c in X.columns if c not in categorical_cols + binary_cols]

print("Catégorielles :", len(categorical_cols))
print("Binaires      :", len(binary_cols))
print("Numériques    :", len(numeric_cols))

Catégorielles : 9
Binaires      : 8
Numériques    : 19


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print("Décrochage train :", y_train.mean().round(3))
print("Décrochage test  :", y_test.mean().round(3))

(3539, 36) (885, 36)
Décrochage train : 0.321
Décrochage test  : 0.321


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ("bin", "passthrough", binary_cols),
])

preprocessor.fit(X_train)                    # apprend sur le TRAIN uniquement
X_train_t = preprocessor.transform(X_train)
X_test_t = preprocessor.transform(X_test)

print("Avant :", X_train.shape[1], "colonnes")
print("Après :", X_train_t.shape[1], "colonnes")

Avant : 36 colonnes
Après : 235 colonnes


In [12]:
import numpy as np
print("Valeurs manquantes après transformation :", np.isnan(X_train_t).sum())
print("Moyenne des variables numériques (train) ~ 0 :",
      X_train_t[:, :len(numeric_cols)].mean().round(3))

Valeurs manquantes après transformation : 0
Moyenne des variables numériques (train) ~ 0 : -0.0


In [13]:
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

SEED = 42

# modèle de base + petite grille d'hyperparamètres (préfixe "clf__" = étape du pipeline)
configs = {
    "Régression logistique": (
        LogisticRegression(max_iter=2000),
        {"clf__C": [0.01, 0.1, 1, 10]},
    ),
    "Arbre de décision": (
        DecisionTreeClassifier(random_state=SEED),
        {"clf__max_depth": [3, 5, 8], "clf__min_samples_leaf": [5, 20]},
    ),
    "k-NN": (
        KNeighborsClassifier(),
        {"clf__n_neighbors": [5, 11, 21]},
    ),
    "Random Forest": (
        RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
        {"clf__max_depth": [8, 16], "clf__min_samples_leaf": [1, 5]},
    ),
    "XGBoost": (
        XGBClassifier(eval_metric="logloss", random_state=SEED, n_jobs=-1),
        {"clf__n_estimators": [200, 400],
         "clf__max_depth": [3, 5],
         "clf__learning_rate": [0.05, 0.1]},
    ),
}

In [14]:
import time

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
best_models, cv_results = {}, {}

for name, (estimator, grid) in configs.items():
    pipe = Pipeline([("prep", clone(preprocessor)), ("clf", estimator)])
    search = GridSearchCV(pipe, grid, scoring="roc_auc", cv=cv, n_jobs=-1)
    t0 = time.time()
    search.fit(X_train, y_train)            # X_train brut : le pipeline s'occupe de tout
    best_models[name] = search.best_estimator_
    cv_results[name] = (search.best_score_, search.best_params_, time.time() - t0)
    print(f"{name:22s} AUC CV = {search.best_score_:.3f} | {search.best_params_} | {time.time()-t0:.0f}s")

Régression logistique  AUC CV = 0.921 | {'clf__C': 1} | 4s
Arbre de décision      AUC CV = 0.881 | {'clf__max_depth': 5, 'clf__min_samples_leaf': 20} | 1s
k-NN                   AUC CV = 0.876 | {'clf__n_neighbors': 21} | 1s
Random Forest          AUC CV = 0.915 | {'clf__max_depth': 16, 'clf__min_samples_leaf': 1} | 9s
XGBoost                AUC CV = 0.918 | {'clf__learning_rate': 0.05, 'clf__max_depth': 3, 'clf__n_estimators': 400} | 10s


In [15]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform

best_models_v1 = dict(best_models)          # copie de secours de la version actuelle

# --- XGBoost : recherche aléatoire, plages plus larges ---
pipe_xgb = Pipeline([("prep", clone(preprocessor)),
                     ("clf", XGBClassifier(eval_metric="logloss", random_state=SEED, n_jobs=1))])
param_xgb = {
    "clf__n_estimators": randint(200, 700),
    "clf__max_depth": randint(2, 6),
    "clf__learning_rate": loguniform(0.01, 0.2),
    "clf__subsample": uniform(0.6, 0.4),          # entre 0,6 et 1,0
    "clf__colsample_bytree": uniform(0.5, 0.5),   # entre 0,5 et 1,0
    "clf__min_child_weight": randint(1, 10),
    "clf__reg_lambda": loguniform(0.5, 20),
}
search_xgb = RandomizedSearchCV(pipe_xgb, param_xgb, n_iter=40, scoring="roc_auc",
                                cv=cv, n_jobs=-1, random_state=SEED)
search_xgb.fit(X_train, y_train)

# --- Régression logistique : grille de C plus fine ---
pipe_lr = Pipeline([("prep", clone(preprocessor)),
                    ("clf", LogisticRegression(max_iter=3000))])
search_lr = GridSearchCV(pipe_lr, {"clf__C": [0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10]},
                         scoring="roc_auc", cv=cv, n_jobs=-1)
search_lr.fit(X_train, y_train)

# --- On garde la nouvelle version seulement si la VALIDATION CROISÉE s'améliore ---
for name, s in [("XGBoost", search_xgb), ("Régression logistique", search_lr)]:
    old, new = cv_results[name][0], s.best_score_
    print(f"{name:22s} AUC CV avant = {old:.4f} | après = {new:.4f}")
    print("   meilleurs paramètres :", s.best_params_)
    if new > old + 0.001:
        best_models[name] = s.best_estimator_
        cv_results[name] = (new, s.best_params_, 0)
        print("   -> nouvelle version retenue")
    else:
        print("   -> gain trop faible, on garde l'ancienne version")

XGBoost                AUC CV avant = 0.9176 | après = 0.9192
   meilleurs paramètres : {'clf__colsample_bytree': np.float64(0.5610439773503366), 'clf__learning_rate': np.float64(0.02907730652732417), 'clf__max_depth': 3, 'clf__min_child_weight': 1, 'clf__n_estimators': 584, 'clf__reg_lambda': np.float64(1.159139775776865), 'clf__subsample': np.float64(0.7708431154505025)}
   -> nouvelle version retenue
Régression logistique  AUC CV avant = 0.9213 | après = 0.9213
   meilleurs paramètres : {'clf__C': 1}
   -> gain trop faible, on garde l'ancienne version


In [16]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

rows = []
for name, model in best_models.items():
    proba_test = model.predict_proba(X_test)[:, 1]
    pred = (proba_test >= 0.5).astype(int)
    rows.append({
        "Modèle": name,
        "AUC train": roc_auc_score(y_train, model.predict_proba(X_train)[:, 1]),
        "AUC CV": cv_results[name][0],
        "AUC test": roc_auc_score(y_test, proba_test),
        "Précision": precision_score(y_test, pred),
        "Rappel": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
    })

comparatif = pd.DataFrame(rows).set_index("Modèle").round(3).sort_values("AUC test", ascending=False)
comparatif

,AUC train,AUC CV,AUC test,Précision,Rappel,F1
Modèle,,,,,,
XGBoost,0.961,0.919,0.936,0.888,0.757,0.817
Régression logistique,0.936,0.921,0.932,0.882,0.761,0.817
Random Forest,0.998,0.915,0.926,0.889,0.708,0.788
k-NN,0.914,0.876,0.887,0.910,0.535,0.674
Arbre de décision,0.901,0.881,0.885,0.804,0.722,0.761


In [17]:
import plotly.graph_objects as go
from sklearn.metrics import roc_curve

probas = {n: m.predict_proba(X_test)[:, 1] for n, m in best_models.items()}

fig = go.Figure()
for n, p in probas.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines",
                             name=f"{n} (AUC={roc_auc_score(y_test, p):.3f})"))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                         line=dict(dash="dash", color="gray"), name="Hasard"))
fig.update_layout(title="Courbes ROC (jeu de test)",
                  xaxis_title="Taux de faux positifs", yaxis_title="Taux de vrais positifs (rappel)")
fig.show()

In [18]:
from sklearn.metrics import precision_recall_curve, average_precision_score

fig = go.Figure()
for n, p in probas.items():
    prec, rec, _ = precision_recall_curve(y_test, p)
    fig.add_trace(go.Scatter(x=rec, y=prec, mode="lines",
                             name=f"{n} (PR-AUC={average_precision_score(y_test, p):.3f})"))
fig.add_hline(y=y_test.mean(), line_dash="dash", line_color="gray",
              annotation_text="Hasard (taux de décrochage)")
fig.update_layout(title="Courbes précision-rappel (jeu de test)",
                  xaxis_title="Rappel", yaxis_title="Précision")
fig.show()

In [19]:
from sklearn.metrics import confusion_matrix
import plotly.express as px

model_name, seuil = "XGBoost", 0.5
pred = (probas[model_name] >= seuil).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

print(f"Vrais positifs (décrocheurs détectés)   : {tp}")
print(f"Faux négatifs (décrocheurs ratés)       : {fn}")
print(f"Faux positifs (fausses alertes)         : {fp}")
print(f"Vrais négatifs (non-décrocheurs OK)     : {tn}")

px.imshow([[tn, fp], [fn, tp]], text_auto=True,
          x=["Prédit : reste", "Prédit : décroche"], y=["Réel : reste", "Réel : décroche"],
          title=f"Matrice de confusion - {model_name} (seuil {seuil})").show()

Vrais positifs (décrocheurs détectés)   : 215
Faux négatifs (décrocheurs ratés)       : 69
Faux positifs (fausses alertes)         : 27
Vrais négatifs (non-décrocheurs OK)     : 574


In [20]:
import numpy as np

p = probas["XGBoost"]
rows = []
for s in np.arange(0.1, 0.91, 0.1):
    pred = (p >= s).astype(int)
    rows.append({"Seuil": round(s, 1),
                 "Précision": precision_score(y_test, pred, zero_division=0),
                 "Rappel": recall_score(y_test, pred),
                 "F1": f1_score(y_test, pred),
                 "Étudiants alertés": int(pred.sum())})
pd.DataFrame(rows).round(3)

,Seuil,Précision,Rappel,F1,Étudiants alertés
0,0.1,0.536,0.951,0.685,504
1,0.2,0.695,0.891,0.781,364
2,0.3,0.786,0.856,0.820,309
3,0.4,0.824,0.806,0.815,278
4,0.5,0.888,0.757,0.817,242
5,0.6,0.914,0.715,0.802,222
6,0.7,0.945,0.662,0.778,199
7,0.8,0.965,0.588,0.731,173
8,0.9,0.992,0.440,0.610,126


In [21]:
print("Avant la fusion :")
print(df["Target"].value_counts())
print(df["Target"].value_counts(normalize=True).round(3))

print("\nAprès la fusion :")
print(df["Dropout"].value_counts())
print("Rapport :", round((df["Dropout"] == 0).sum() / (df["Dropout"] == 1).sum(), 2), "pour 1")

print("\nTrain :", y_train.value_counts().to_dict())
print("Test  :", y_test.value_counts().to_dict())

Avant la fusion :
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64
Target
Graduate    0.499
Dropout     0.321
Enrolled    0.179
Name: proportion, dtype: float64

Après la fusion :
Dropout
0    3003
1    1421
Name: count, dtype: int64
Rapport : 2.11 pour 1

Train : {0: 2402, 1: 1137}
Test  : {0: 601, 1: 284}


In [22]:
from sklearn.metrics import accuracy_score, average_precision_score

rows = []
for name, model in best_models.items():
    p = model.predict_proba(X_test)[:, 1]
    pred = (p >= 0.5).astype(int)
    rows.append({"Modèle": name,
                 "AUC": roc_auc_score(y_test, p),
                 "PR-AUC": average_precision_score(y_test, p),
                 "Accuracy": accuracy_score(y_test, pred),
                 "Précision": precision_score(y_test, pred),
                 "Rappel": recall_score(y_test, pred),
                 "F1": f1_score(y_test, pred)})
pd.DataFrame(rows).set_index("Modèle").round(3).sort_values("AUC", ascending=False)

,AUC,PR-AUC,Accuracy,Précision,Rappel,F1
Modèle,,,,,,
XGBoost,0.936,0.909,0.892,0.888,0.757,0.817
Régression logistique,0.932,0.903,0.890,0.882,0.761,0.817
Random Forest,0.926,0.895,0.878,0.889,0.708,0.788
k-NN,0.887,0.828,0.834,0.910,0.535,0.674
Arbre de décision,0.885,0.835,0.854,0.804,0.722,0.761


In [23]:
import joblib, json, os, shutil, sklearn, xgboost, pandas

os.makedirs("artifacts", exist_ok=True)

# les 5 modèles (pipeline = préprocesseur + modèle), compressés pour limiter la taille
noms_fichiers = {
    "XGBoost": "xgboost",
    "Régression logistique": "logreg",
    "Random Forest": "random_forest",
    "k-NN": "knn",
    "Arbre de décision": "decision_tree",
}
for nom, fichier in noms_fichiers.items():
    joblib.dump(best_models[nom], f"artifacts/{fichier}_pipeline.joblib", compress=3)

# le jeu de test (pour l'API et pour choisir un étudiant plus tard)
test_df = X_test.copy()
test_df["Dropout"] = y_test
test_df.to_csv("artifacts/test_set.csv", index=False)

# informations dont l'API aura besoin
meta = {
    "target": "Dropout",
    "features": list(X.columns),
    "categorical": categorical_cols,
    "binary": binary_cols,
    "numeric": numeric_cols,
    "default_threshold": 0.3,
}
json.dump(meta, open("artifacts/meta.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

shutil.make_archive("artifacts", "zip", "artifacts")

for f in sorted(os.listdir("artifacts")):
    print(f"{f:35s} {os.path.getsize(f'artifacts/{f}') / 1e6:.1f} Mo")
print("scikit-learn :", sklearn.__version__)
print("xgboost      :", xgboost.__version__)
print("pandas       :", pandas.__version__)

decision_tree_pipeline.joblib       0.0 Mo
knn_pipeline.joblib                 0.4 Mo
logreg_pipeline.joblib              0.0 Mo
meta.json                           0.0 Mo
random_forest_pipeline.joblib       2.8 Mo
test_set.csv                        0.1 Mo
xgboost_pipeline.joblib             0.1 Mo
scikit-learn : 1.6.1
xgboost      : 3.4.1
pandas       : 2.3.3
